In [30]:
!pip -q install torch torch_geometric


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, MessagePassing
from torch_geometric.utils import add_self_loops
import torch_geometric.transforms as T

In [32]:
# --------------------------
# 0) Config y reproducibilidad
# --------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Hiperparámetros
HIDDEN_DIM = 16
DROPOUT    = 0.5
LR         = 0.01
WEIGHT_DEC = 5e-4
EPOCHS     = 200  # máximo de épocas
USE_EARLY_STOPPING = False
PATIENCE   = 20   # épocas sin mejora en validación antes de detenerse

Device: cpu


In [33]:
# --------------------------
# 1) Dataset: Cora (Planetoid)
#    Incluye máscaras de train/val/test
#    Normalizamos features por nodo
# --------------------------
root = "data/Planetoid"
dataset = Planetoid(root=root, name="Cora", transform=T.NormalizeFeatures())
data = dataset[0].to(device)  # Un único grafo con x, edge_index, y, train_mask, val_mask, test_mask

print(f"#Nodos: {data.num_nodes}  | #Aristas: {data.num_edges}  | #Features: {dataset.num_features}  | #Clases: {dataset.num_classes}")
print(f"Máscaras -> train: {int(data.train_mask.sum())}, val: {int(data.val_mask.sum())}, test: {int(data.test_mask.sum())}")

#Nodos: 2708  | #Aristas: 10556  | #Features: 1433  | #Clases: 7
Máscaras -> train: 140, val: 500, test: 1000


In [34]:
# --------------------------
# 2) Modelo GCN (2 capas)
# --------------------------
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(dataset.num_features, HIDDEN_DIM, dataset.num_classes, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DEC)
criterion = nn.CrossEntropyLoss()

In [35]:
# --------------------------
# 3) Accuracy en máscaras (nodos)
# --------------------------
@torch.no_grad()
def masked_accuracy(logits, y, mask):
    preds = logits.argmax(dim=1)
    correct = (preds[mask] == y[mask]).sum().item()
    total = int(mask.sum())
    return 100.0 * correct / total

In [36]:
# --------------------------
# 4) Entrenamiento con early stopping
# --------------------------
best_val_acc, best_state = float('-inf'), None
best_epoch, epochs_without_improvement = 0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)          # [N, C]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    # Eval
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        train_acc = masked_accuracy(logits, data.y, data.train_mask)
        val_acc   = masked_accuracy(logits, data.y, data.val_mask)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"[{epoch:03d}] loss={loss.item():.4f} | train={train_acc:.2f}% | val={val_acc:.2f}%")

    if USE_EARLY_STOPPING and epochs_without_improvement >= PATIENCE:
        print(f"Early stopping en época {epoch}: sin mejora en validación durante {PATIENCE} épocas.")
        break

# Restaurar el modelo seleccionado con validación y evaluarlo una sola vez en test.
model.load_state_dict(best_state)
model.to(device).eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    test_acc = masked_accuracy(logits, data.y, data.test_mask)

print(f"Mejor época: {best_epoch} | mejor val: {best_val_acc:.2f}% | test: {test_acc:.2f}%")


[001] loss=1.9459 | train=32.14% | val=14.40%
[020] loss=1.7174 | train=85.71% | val=52.60%
[040] loss=1.3281 | train=96.43% | val=70.40%
[060] loss=0.9540 | train=99.29% | val=76.00%
[080] loss=0.6925 | train=99.29% | val=77.60%
[100] loss=0.5591 | train=99.29% | val=76.60%
[120] loss=0.4700 | train=99.29% | val=77.40%
[140] loss=0.4071 | train=99.29% | val=77.80%
[160] loss=0.3849 | train=100.00% | val=77.80%
[180] loss=0.3435 | train=100.00% | val=77.80%
[200] loss=0.3223 | train=100.00% | val=78.00%
Mejor época: 125 | mejor val: 78.40% | test: 80.60%


Con early stopping:
[001] loss=1.9459 | train=32.14% | val=14.40%
[020] loss=1.7174 | train=85.71% | val=52.60%
[040] loss=1.3281 | train=96.43% | val=70.40%
[060] loss=0.9540 | train=99.29% | val=76.00%
[080] loss=0.6925 | train=99.29% | val=77.60%
[100] loss=0.5591 | train=99.29% | val=76.60%
Early stopping en época 100: sin mejora en validación durante 20 épocas.
Mejor época: 80 | mejor val: 77.60% | test: 79.00%

Sin early stopping:

[020] loss=1.7174 | train=85.71% | val=52.60%
[040] loss=1.3281 | train=96.43% | val=70.40%
[060] loss=0.9540 | train=99.29% | val=76.00%
[080] loss=0.6925 | train=99.29% | val=77.60%
[100] loss=0.5591 | train=99.29% | val=76.60%
[120] loss=0.4700 | train=99.29% | val=77.40%
[140] loss=0.4071 | train=99.29% | val=77.80%
[160] loss=0.3849 | train=100.00% | val=77.80%
[180] loss=0.3435 | train=100.00% | val=77.80%
[200] loss=0.3223 | train=100.00% | val=78.00%
Mejor época: 125 | mejor val: 78.40% | test: 80.60%

Ejercicios:
1) Agregar early stopping.
2) Comparar GCNConv con GraphSAGE, Message Passing y GAT.
3) Comparar la performance en distintos datasets: Cora, CiteSeer, PubMed, etc.

In [37]:
# Ejercicio 2: comparar GCN, GraphSAGE, Message Passing y GAT
# Todos usan la misma división train/val/test e hiperparámetros.

class MeanMessagePassing(MessagePassing):
    # Message passing explícito: transforma, promedia mensajes y actualiza.
    def __init__(self, in_dim, out_dim):
        super().__init__(aggr='mean')
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        return self.propagate(edge_index, x=self.lin(x))

    def message(self, x_j):
        return x_j


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


class MPNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = MeanMessagePassing(in_dim, hidden_dim)
        self.conv2 = MeanMessagePassing(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def train_and_evaluate(model, name):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DEC)
    best_val_acc, best_state = float('-inf'), None
    best_epoch, epochs_without_improvement = 0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            val_acc = masked_accuracy(logits, data.y, data.val_mask)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if USE_EARLY_STOPPING and epochs_without_improvement >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        test_acc = masked_accuracy(logits, data.y, data.test_mask)

    return {'modelo': name, 'mejor_epoca': best_epoch, 'val_acc': best_val_acc, 'test_acc': test_acc}


# Reiniciamos la semilla antes de crear cada arquitectura para resultados reproducibles.
def make_model(model_class):
    torch.manual_seed(SEED)
    return model_class(dataset.num_features, HIDDEN_DIM, dataset.num_classes, DROPOUT)

models = {
    'GCN': make_model(GCN),
    'GraphSAGE': make_model(GraphSAGE),
    'Message Passing (mean)': make_model(MPNN),
    'GAT (8 heads)': make_model(GAT),
}

results = [train_and_evaluate(model, name) for name, model in models.items()]
for result in sorted(results, key=lambda x: x['test_acc'], reverse=True):
    print(f"{result['modelo']:<24} | época: {result['mejor_epoca']:>3} | val: {result['val_acc']:6.2f}% | test: {result['test_acc']:6.2f}%")

# Interpretación: GCN normaliza por grado; GraphSAGE separa nodo y vecinos;
# MPNN expone message -> aggregate -> update; GAT aprende la importancia de cada vecino.
# Elegí por val_acc; usá test_acc únicamente como evaluación final.

GAT (8 heads)            | época:  19 | val:  82.00% | test:  82.30%
Message Passing (mean)   | época: 108 | val:  80.20% | test:  81.10%
GraphSAGE                | época: 158 | val:  78.20% | test:  80.00%
GCN                      | época: 152 | val:  78.80% | test:  79.90%


## Ejercicio 3: GCN en distintos datasets

Se mantiene exactamente la misma arquitectura GCN y los mismos hiperparámetros. Cada dataset aporta sus propias máscaras de entrenamiento, validación y test.

In [38]:
DATASET_NAMES = ['Cora', 'CiteSeer', 'PubMed']


def train_gcn_on_dataset(dataset_name):
    # Planetoid descarga el dataset si todavía no está disponible localmente.
    current_dataset = Planetoid(
        root='data/Planetoid', name=dataset_name, transform=T.NormalizeFeatures()
    )
    current_data = current_dataset[0].to(device)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    model = GCN(
        current_dataset.num_features, HIDDEN_DIM, current_dataset.num_classes, DROPOUT
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DEC)
    best_val_acc, best_state = float('-inf'), None
    best_epoch, epochs_without_improvement = 0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(current_data.x, current_data.edge_index)
        loss = criterion(logits[current_data.train_mask], current_data.y[current_data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(current_data.x, current_data.edge_index)
            val_acc = masked_accuracy(logits, current_data.y, current_data.val_mask)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if USE_EARLY_STOPPING and epochs_without_improvement >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(current_data.x, current_data.edge_index)
        train_acc = masked_accuracy(logits, current_data.y, current_data.train_mask)
        test_acc = masked_accuracy(logits, current_data.y, current_data.test_mask)

    return {
        'dataset': dataset_name,
        'nodos': current_data.num_nodes,
        'aristas': current_data.num_edges,
        'features': current_dataset.num_features,
        'clases': current_dataset.num_classes,
        'mejor_epoca': best_epoch,
        'train_acc': train_acc,
        'val_acc': best_val_acc,
        'test_acc': test_acc,
    }


gcn_dataset_results = [train_gcn_on_dataset(name) for name in DATASET_NAMES]
for result in sorted(gcn_dataset_results, key=lambda x: x['test_acc'], reverse=True):
    print(
        f"{result['dataset']:<9} | nodos: {result['nodos']:>5} | aristas: {result['aristas']:>6} | "
        f"mejor época: {result['mejor_epoca']:>3} | train: {result['train_acc']:6.2f}% | "
        f"val: {result['val_acc']:6.2f}% | test: {result['test_acc']:6.2f}%"
    )


Processing...
Done!
Processing...
Done!


Cora      | nodos:  2708 | aristas:  10556 | mejor época: 125 | train:  99.29% | val:  78.40% | test:  80.60%
PubMed    | nodos: 19717 | aristas:  88648 | mejor época: 111 | train:  98.33% | val:  81.40% | test:  78.50%
CiteSeer  | nodos:  3327 | aristas:   9104 | mejor época:  59 | train:  93.33% | val:  73.60% | test:  71.00%


### Interpretación

La comparación mide cuán bien generaliza la **misma GCN** sobre grafos con distinto tamaño, vocabulario, cantidad de clases y estructura de citas. Compará principalmente `test_acc`, pero seleccioná épocas e hiperparámetros a partir de `val_acc`. Una diferencia de performance no implica por sí sola que un dataset sea más fácil: también influyen sus máscaras, conectividad y distribución de clases.